In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
sys.path.append('../../')
print(sys.path)


import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrames

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

# We only need the categories for this study and the expandable systs are really big
cols_to_keep = [
    ('nu_categ', '', '', ''),
    ('genie_categ', '', '', ''),
    ('nu_categ_proton_reduced', '', '', '')
]
mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load in time cosmic df
mc_in_time_cosmics_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_in_time_cosmics.df", keys2load, 100)
mc_in_time_cosmics_evt_df = mc_in_time_cosmics_df['cc1pi']
mc_in_time_cosmics_hdr_df = mc_in_time_cosmics_df['hdr']

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

In [ ]:

#Load CV lowE dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_lowE_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_lowE_CV.df", keys2load, 5, filter_df = False)
mc_bnb_lowE_evt_df = mc_bnb_lowE_df['cc1pi']
mc_bnb_lowE_nu_df = mc_bnb_lowE_df['nudf']
mc_bnb_lowE_hdr_df = mc_bnb_lowE_df['hdr']
mc_bnb_lowE_nu_df = mc_bnb_lowE_nu_df[cols_to_keep]

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))


#Low E
mc_bnb_lowE_tot_pot = mc_bnb_lowE_hdr_df['pot'].sum()
print("dirt_tot_pot: %.3e" %(mc_bnb_lowE_tot_pot))
mc_bnb_lowE_pot_scale = data_tot_pot / mc_bnb_lowE_tot_pot
print("dirt_pot_scale: %.3e" %(mc_bnb_lowE_pot_scale))
mc_bnb_lowE_evt_df[pot_weight_col] = mc_bnb_lowE_pot_scale * np.ones(len(mc_bnb_lowE_evt_df))




intime_gates = mc_in_time_cosmics_hdr_df[mc_in_time_cosmics_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics data gates: {:.2e}".format(intime_gates))
f = 0.075
scale_intime_to_lightdata = (1-f)*data_gates/intime_gates
print("goal scale: {:.2f}".format(scale_intime_to_lightdata))
mc_in_time_cosmics_evt_df[pot_weight_col] = scale_intime_to_lightdata * np.ones(len(mc_in_time_cosmics_evt_df))

In [ ]:
mc_evt_df = mc_bnb_evt_df
if "ar23p" in bnb_path:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)   
print("Finished loading")    

mc_in_time_cosmics_evt_df = perform_truth_matching(mc_in_time_cosmics_evt_df, mc_bnb_nu_df.head(1))   
print("TM for cosmic done")

mc_evt_df = concat_shift_first_index(mc_evt_df,mc_in_time_cosmics_evt_df)
mc_bnb_lowE_evt_df = perform_truth_matching(mc_bnb_lowE_evt_df, mc_bnb_lowE_nu_df)
print("TM for BNB nu df")
mc_evt_df = concat_shift_first_index(mc_evt_df,mc_bnb_lowE_evt_df)

# Test background composition

In [ ]:
cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")

#mc_cumulative_masks_sideband_pion = build_event_cumulative_masks(slc_df, sideband = "two_pions")
#mc_cumulative_masks_sideband_proton = build_event_cumulative_masks(slc_df, sideband = "proton")
#mc_cumulative_masks_sideband = mc_cumulative_masks_sideband_pion | mc_cumulative_masks_sideband_proton

In [ ]:
slc_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
        )    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tarfile


reweigth_pot = 1e20
scale = reweigth_pot/data_tot_pot


# --- CONFIGURATION TOGGLES ---
HIDE_COSMIC_BAR = False  
USE_LOG_SCALE = True
X_MIN_LOG = 1000 
# ----------------------------

plot_data = []
for stage_name, mask in cumulative_masks.items():
    if HIDE_COSMIC_BAR and stage_name == "cosmic":
        continue

    # 1. Collapse the 4-level mask into a 3-level mask
    # This keeps a slice if ANY of its PFPs are True in the mask
    collapsed_mask = mask.groupby(['__ntuple', 'entry', 'rec.slc..index']).first()

    # 2. Align the collapsed mask with slc_df
    # We use reindex to ensure the rows match slc_df exactly, 
    # filling missing values with False.
    final_mask = collapsed_mask.reindex(slc_df.index, fill_value=False)

    # 3. Filter and Aggregate
    filtered_categs = slc_df.loc[final_mask, [('truth','nu_categ','','','',''), pot_weight_col]]
    counts = filtered_categs.groupby([('truth','nu_categ','','','','')])[[pot_weight_col]].sum()*scale

    counts = counts.iloc[:, 0]
    counts.name = stage_name
    plot_data.append(counts)

# Create the initial DataFrame
df_results = pd.concat(plot_data, axis=1).T.fillna(0)

# Ensure CC1pi is the first column before renaming
signal_tag = 'CC1pi'
if signal_tag in df_results.columns:
    cols = [signal_tag] + [c for c in df_results.columns if c != signal_tag]
    df_results = df_results[cols]

# 2. MATCH COLORS TO CURRENT COLUMNS
# We do this BEFORE renaming so we can use the original tags as keys
plot_colors = [category_colors.get(col, "#000000") for col in df_results.columns]

# --- 1. Prepare Data for Plotting ---
# (Assuming df_results and plot_colors are already defined from your previous steps)
df_results_renamed = df_results.rename(columns=bkg_name_nice_map).rename(index=cut_name_nice_map)

# Create the side-by-side figure
# figsize=(20, 8) ensures they are wide enough to sit next to each other
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8), sharey=True)

# --- 2. Efficiency Plot (Left) ---
df_plot_efficiency = df_results_renamed[df_results_renamed.sum(axis=1) > 0]

df_plot_efficiency.plot(
    kind='barh',
    stacked=True,
    ax=ax1,
    color=plot_colors,
    legend=True # Keep legend on for the first plot
)

if USE_LOG_SCALE:
    ax1.set_xscale('log')
    # Add extra headroom (multiplier) so the bars don't hit the right edge
    max_val = df_plot_efficiency.sum(axis=1).max() * 5 
    ax1.set_xlim(X_MIN_LOG, max_val)

ax1.set_title(r"$\nu\mu$CC1$\pi$ selection efficiency", fontsize=20, pad=15)
ax1.set_xlabel("Candidate Slices", fontsize=18)
ax1.set_ylabel("") # sharey=True handles the labels
ax1.grid(True, which="both", ls="-", alpha=0.2)
ax1.tick_params(axis='both', labelsize=16)

# Move Legend INSIDE the Efficiency plot
ax1.legend(
    loc='upper right',
    fontsize=18, 
    framealpha=1.0, 
    edgecolor='black', 
    fancybox=False
)

# --- 3. Purity Plot (Right) ---
# Normalize to fractions
df_purity = df_results_renamed.div(df_results_renamed.sum(axis=1), axis=0)

df_purity.plot(
    kind='barh',
    stacked=True,
    ax=ax2,
    color=plot_colors,
    legend=False # Hide legend on the second plot to avoid redundancy
)

ax2.set_title(r"$\nu\mu$CC1$\pi$ selection purity", fontsize=20, pad=15)
ax2.set_xlabel("Fraction of Total Slices", fontsize=18)
ax2.set_ylabel("")
ax2.tick_params(axis='both', labelsize=16)

# Purity-specific styling
ax2.set_xlim(0, 1)
ticks = np.arange(0, 1.1, 0.1)
ax2.set_xticks(ticks)
ax2.grid(True, axis='x', ls='--', alpha=0.6, color='gray', zorder=0)

# Final Layout Adjustments
plt.tight_layout()
# Adjust wspace to bring the plots closer since they share a Y-axis
plt.subplots_adjust(wspace=0.08) 

plt.show()

parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/PurEffGraphs"
if not os.path.exists(parent_path):
    os.makedirs(parent_path)

# Define the specific filename (e.g., based on a variable or timestamp)
f_name = "cc1pi_selection_summary.pdf" 
save_full_path = os.path.join(parent_path, f_name)

# 2. Save the Figure (Captures both side-by-side plots)
fig.savefig(save_full_path, format='pdf', bbox_inches='tight')

# 3. Create the Tarball
tar_output_path = f"{parent_path}.tar" # Added .gz to match "w:gz" mode
with tarfile.open(tar_output_path, "w:gz") as tar:
    # arcname ensures the folder structure inside the tar is clean
    tar.add(parent_path, arcname=os.path.basename(parent_path))

print(f"Plot saved to {save_full_path} and bundled into {tar_output_path}")


In [ ]:

print(scale)

# Assuming 'CC1Pi' is your signal column
signal_col = '$\\nu_{\\mu}$CC1$\\pi^{\\pm}$'
#signal_col = "$\\nu_{\\mu}$CC0$\\pi^{\\pm}$1p"

# 1. Calculate Total Events per stage
total_per_stage = df_results_renamed.sum(axis=1)

# 2. Calculate Purity: Signal / Total at each stage
purity = (df_results_renamed[signal_col] / total_per_stage) * 100

# 3. Calculate Efficiency: Signal at stage / Signal at first stage
# (Recreating num_events_0 logic)
initial_signal_count = df_results_renamed[signal_col].iloc[0]
efficiency = (df_results_renamed[signal_col] / initial_signal_count) * 100

# 4. Create a summary table for easy viewing
summary_df = pd.DataFrame({
    'Signal_Events': df_results_renamed[signal_col]*scale,
    'Total_Events': total_per_stage*scale,
    'Purity (%)': purity,
    'Efficiency (%)': efficiency
})

print(summary_df)

In [ ]:
import numpy as np

print(f"{'Cut Name':<20} | {'Purity':<10} | {'Efficiency':<10} | {'S/sqrt(B)':<10} | {'pur * eff':<10}")
print("-" * 75)

for stage in df_results_renamed.index:
    n_signal = df_results_renamed.loc[stage, signal_col]*scale
    n_total = total_per_stage.loc[stage]*scale
    n_bkg = n_total - n_signal
    
    pur = (n_signal / n_total) * 100 if n_total > 0 else 0
    eff = (n_signal / initial_signal_count) * 100
    pur_eff = pur*eff/100
    s_sqrt_b = n_signal / np.sqrt(n_bkg) if n_bkg > 0 else np.nan
    
    print(f"{stage:<20} | {pur:>8.2f}% | {eff:>8.2f}% | {s_sqrt_b:>8.2f} | {pur_eff:8.2f}") 

In [ ]:
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first()], ('truth','genie_categ','','','',''))

In [ ]:
print("0p")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first() & mask_dict["0p"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))
print("1p")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first() & mask_dict["1p"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))
print("2p+")
HelperFunctions.print_purity(slc_df[cumulative_masks["energy"].groupby(['__ntuple', 'entry', 'rec.slc..index']).first() & mask_dict["2plusp"](slc_df)], ('truth','nu_categ_proton_reduced','','','',''))